# Convolutional Autoencoder on MNIST

An autoencoder learns a compressed representation of the input and then reconstructs it. This notebook trains a small convolutional autoencoder on MNIST and visualises the reconstructed digits.

## Imports and data

We load the MNIST dataset. For an autoencoder we don't need labels — the goal is to reconstruct the input image itself, so the images serve as both input and target.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = Path("../../data/raw")

transform = transforms.ToTensor()
train_ds = datasets.MNIST(data_dir, train=True, download=True, transform=transform)
test_ds = datasets.MNIST(data_dir, train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Device: {device}")

## Define autoencoder

The autoencoder has two parts: an **encoder** that compresses the 28×28 image into a smaller latent representation using strided convolutions, and a **decoder** that reconstructs the image back to its original size using transposed convolutions. The bottleneck forces the network to learn the most important features.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),  # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # 14x14 -> 7x7
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),  # 7x7 -> 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3, stride=2, padding=1, output_padding=1),  # 14x14 -> 28x28
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = Autoencoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

model

## Train

We train for 10 epochs minimising MSE (mean squared error) between the original and reconstructed images. Labels are ignored — the model learns to reproduce its own input.

In [ ]:
history = []

for epoch in range(1, 11):
    model.train()
    total_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        reconstructed = model(images)
        loss = loss_fn(reconstructed, images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    history.append(avg_loss)
    print(f"Epoch {epoch:2d}: reconstruction_loss={avg_loss:.5f}")

## Reconstructions

We pass test images through the trained autoencoder and compare originals (top row) with reconstructions (bottom row). Good reconstructions indicate the model has learned meaningful features in its compressed representation.

In [ ]:
model.eval()
sample, _ = next(iter(test_loader))
sample = sample[:8].to(device)

with torch.no_grad():
    recon = model(sample).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(sample[i].cpu().squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Reconstructed")
plt.tight_layout()
plt.show()

## Training curve

We plot the reconstruction loss over epochs. A steadily decreasing MSE shows the autoencoder is learning to compress and reconstruct the digits more accurately.

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(range(1, len(history) + 1), history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Reconstruction loss")
plt.tight_layout()
plt.show()

## Practical note

Autoencoders learn unsupervised compressed representations. They are useful for dimensionality reduction, denoising, and anomaly detection. The bottleneck size controls how much information is retained.